In [3]:
from lczero.backends import Weights, Backend, GameState
w = Weights('models/weights-256_10.pb')


In [ ]:
print(w.filters())

: 

In [4]:
Backend.available_backends()

['eigen',
 'trivial',
 'random',
 'check',
 'recordreplay',
 'roundrobin',
 'multiplexing',
 'demux']

In [5]:
b = Backend(weights=w)

Creating backend [eigen]...
Using Eigen version 3.4.0
Eigen max batch size is 256.


In [7]:
args = [el for el in dir(b) if not el.startswith('__')]
print(args)

['available_backends', 'capabilities', 'evaluate']


In [6]:
g = GameState(moves=['e2e4', 'e7e5'])
print(g.as_string())

https://lc0.org/fen/rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR_w_KQkq_-


In [8]:
def dir_c(a):
    return [item for item in dir(a) if not item.startswith('__')]

In [ ]:
!pip install chess

In [25]:
import chess
board = chess.Board("rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1")
board.push_uci("e7e5")

Move.from_uci('e7e5')

In [28]:
input_planes = b.GetNetworkInput(board.fen())

AttributeError: 'backends.Backend' object has no attribute 'GetNetworkInput'

In [ ]:
# 1. Создание первого входа из объекта g (предполагается, что g и b уже определены ранее)
i1 = g.as_input(b)

# 2. Создание второго входа из новой позиции FEN (исправлено: добавлен аргумент backend 'b')
i2 = GameState(fen='2R5/5kpp/4p3/p4p2/3B4/1K5N/4rNPP/8 b - - 0 29').as_input(b)

# 3. Проверка маски для первого состояния (вывод в консоль)
print(bin(i1.mask(0)))

# 4. Проверка маски для второго состояния (вывод в консоль)
print(bin(i2.mask(0)))

# 5. Оценка обоих состояний движком
o1, o2 = b.evaluate(i1, i2)

# 6. Вывод оценки качества (Q-value) для первой позиции
print(o1.q())

# 7. Вывод оценки качества (Q-value) для второй позиции
print(o2.q())

# 8. Получение списка ходов и вероятностей политики (softmax) для первой позиции
# Исправлено: использован объект 'g' вместо несуществующего 'p1'
policy_data = list(zip(g.moves(), o1.p_softmax(*g.policy_indices())))

# 9. Вывод результатов (ход -> вероятность)
for move, prob in policy_data:
    print(f"{move}: {prob}")

In [ ]:
print(dir_c(i2))
print()

['clone', 'mask', 'set_mask', 'set_val', 'val']
None


In [13]:
!unzip tf_model.zip

Archive:  tf_model.zip
   creating: tf_model/assets/
  inflating: tf_model/keras_metadata.pb  
  inflating: tf_model/saved_model.pb  
   creating: tf_model/variables/
  inflating: tf_model/variables/variables.data-00000-of-00001  
  inflating: tf_model/variables/variables.index  


In [1]:
import tensorflow as tf


2026-02-25 10:11:03.922268: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-25 10:11:04.804309: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-25 10:11:08.242243: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
try:
    model = tf.saved_model.load('tf_model')
except Exception as e:
    print(f"Exception occured: \n\n {e}")

Instructions for updating:
Use `tf.saved_model.load` instead.
Exception occured: 

 load() missing 2 required positional arguments: 'tags' and 'export_dir'


In [8]:
import tensorflow.compat.v1 as tf1
import os
import glob

def load_weights_from_checkpoint(path):
    print(f"Попытка загрузки весов из чекпоинта в: {path}")
    
    # 1. Ищем файл .index внутри папки variables
    var_dir = os.path.join(path, "variables")
    if not os.path.exists(var_dir):
        print(f"Ошибка: Папка {var_dir} не найдена.")
        return {}
    
    index_files = glob.glob(os.path.join(var_dir, "*.index"))
    if not index_files:
        print("Ошибка: Файл .index не найден. Модель может быть заморожена иначе.")
        return {}
    
    checkpoint_path = index_files[0].replace(".index", "") # Убираем расширение для пути
    print(f"Найден чекпоинт: {checkpoint_path}")
    
    weights_data = {}
    
    # 2. Используем tf.train.NewCheckpointReader для чтения без создания графа
    reader = tf1.train.NewCheckpointReader(checkpoint_path)
    
    # Получаем карту всех тензоров: {имя: форма}
    var_to_shape_map = reader.get_variable_to_shape_map()
    
    print(f"\nНайдено тензоров в чекпоинте: {len(var_to_shape_map)}")
    
    count = 0
    for name, shape in var_to_shape_map.items():
        # Фильтр: пропускаем служебные счетчики шагов (обычно 'global_step')
        if "global_step" in name:
            continue
            
        try:
            tensor = reader.get_tensor(name)
            weights_data[name] = tensor
            
            # Вывод первых 10 для проверки
            if count < 15:
                print(f"Имя: {name:<50} Форма: {str(shape):<20} Тип: {tensor.dtype}")
                # Можно вывести немного данных для уверенности
                # print(f"  Пример: {tensor.flatten()[:3]}")
            count += 1
        except Exception as e:
            print(f"Ошибка чтения {name}: {e}")
            
    if count > 15:
        print(f"... и еще {count - 15} тензоров.")
        
    return weights_data

# Запуск
weights = load_weights_from_checkpoint('tf_model')

if weights:
    print("\nУСПЕХ! Веса загружены в словарь.")
    print("Теперь вы можете создать Keras модель и присвоить веса вручную.")
else:
    print("\nНе удалось загрузить веса из чекпоинта.")

Попытка загрузки весов из чекпоинта в: tf_model
Найден чекпоинт: tf_model/variables/variables

Найдено тензоров в чекпоинте: 93
Имя: layer_with_weights-9/bias/.ATTRIBUTES/VARIABLE_VALUE Форма: [256]                Тип: float32
Имя: layer_with_weights-7/bias/.ATTRIBUTES/VARIABLE_VALUE Форма: [256]                Тип: float32
Имя: layer_with_weights-6/kernel/.ATTRIBUTES/VARIABLE_VALUE Форма: [3, 3, 256, 256]     Тип: float32
Имя: layer_with_weights-5/bias/.ATTRIBUTES/VARIABLE_VALUE Форма: [256]                Тип: float32
Имя: layer_with_weights-44/bias/.ATTRIBUTES/VARIABLE_VALUE Форма: [1858]               Тип: float32
Имя: layer_with_weights-42/kernel/.ATTRIBUTES/VARIABLE_VALUE Форма: [1, 1, 256, 32]      Тип: float32
Имя: layer_with_weights-42/bias/.ATTRIBUTES/VARIABLE_VALUE Форма: [32]                 Тип: float32
Имя: layer_with_weights-41/kernel/.ATTRIBUTES/VARIABLE_VALUE Форма: [1, 1, 256, 32]      Тип: float32
Имя: layer_with_weights-41/bias/.ATTRIBUTES/VARIABLE_VALUE Форма: [32]

In [38]:
def dir_c(it):
    return [item for item in dir(it) if not item.startswith('_')]
print(dir_c(model))
print(model.signatures)

['call_and_return_all_conditional_losses', 'graph_debug_info', 'keras_api', 'layer-0', 'layer-1', 'layer-10', 'layer-11', 'layer-12', 'layer-13', 'layer-14', 'layer-15', 'layer-16', 'layer-17', 'layer-18', 'layer-19', 'layer-2', 'layer-20', 'layer-21', 'layer-22', 'layer-23', 'layer-24', 'layer-25', 'layer-26', 'layer-27', 'layer-28', 'layer-29', 'layer-3', 'layer-30', 'layer-31', 'layer-32', 'layer-33', 'layer-34', 'layer-35', 'layer-36', 'layer-37', 'layer-38', 'layer-39', 'layer-4', 'layer-40', 'layer-41', 'layer-42', 'layer-43', 'layer-44', 'layer-45', 'layer-46', 'layer-47', 'layer-48', 'layer-49', 'layer-5', 'layer-50', 'layer-51', 'layer-52', 'layer-53', 'layer-54', 'layer-55', 'layer-56', 'layer-57', 'layer-58', 'layer-59', 'layer-6', 'layer-60', 'layer-61', 'layer-62', 'layer-63', 'layer-64', 'layer-65', 'layer-66', 'layer-67', 'layer-68', 'layer-69', 'layer-7', 'layer-70', 'layer-71', 'layer-72', 'layer-73', 'layer-74', 'layer-75', 'layer-76', 'layer-77', 'layer-78', 'layer-7

In [10]:
!unzip tf_model.zip 

Archive:  tf_model.zip
   creating: tf_model/assets/
  inflating: tf_model/keras_metadata.pb  
  inflating: tf_model/saved_model.pb  
   creating: tf_model/variables/
  inflating: tf_model/variables/variables.data-00000-of-00001  
  inflating: tf_model/variables/variables.index  


In [ ]:
import tensorflow.keras as keras
off_mod = keras.models.load_model('tf_model')

ModuleNotFoundError: No module named 'tensorflow.compat.v1.keras'